In [ ]:
"""
TO DO:

Data Engineering:
1. Labeled Datasets -> Different Folders of Good Shot Forms and Bad Shot Forms
2. Impute Data -> Mean of last known value and future known value for Joint Positions, Vel, Acc, Angle.
3. Start and End time's of shot motion -> Keep in mind: ball pos. values are bad so if I want to adjust start/stop based on
position of ball from hand, need to keep in mind
    -Solution: Set all values of 0 to NaN. If val != NaN and value far from hand = Ball has been shot
4. Deal with different length of shot videos. Padding?

Model Engineering:
1. LSTM Model

""

In [20]:
import pandas as pd
import os

datasets_folder = 'Datasets'

Datasets = [os.path.join(datasets_folder, file) for file in os.listdir(datasets_folder) if file.endswith(".csv")]

data_frames = []
for data_set in Datasets:
    df = pd.read_csv(data_set)
    data_frames.append(df)

print(data_frames[0].columns.tolist())

['video', 'frame', 'label', 'sports_ball_positions', 'LEFT_SHOULDER_vel', 'RIGHT_SHOULDER_vel', 'LEFT_SHOULDER_acc', 'RIGHT_SHOULDER_acc', 'LEFT_ELBOW_vel', 'RIGHT_ELBOW_vel', 'LEFT_ELBOW_acc', 'RIGHT_ELBOW_acc', 'LEFT_WRIST_vel', 'RIGHT_WRIST_vel', 'LEFT_WRIST_acc', 'RIGHT_WRIST_acc', 'LEFT_HIP_vel', 'RIGHT_HIP_vel', 'LEFT_HIP_acc', 'RIGHT_HIP_acc', 'LEFT_SHOULDER_pos_x', 'LEFT_SHOULDER_pos_y', 'RIGHT_SHOULDER_pos_x', 'RIGHT_SHOULDER_pos_y', 'LEFT_ELBOW_pos_x', 'LEFT_ELBOW_pos_y', 'RIGHT_ELBOW_pos_x', 'RIGHT_ELBOW_pos_y', 'LEFT_WRIST_pos_x', 'LEFT_WRIST_pos_y', 'RIGHT_WRIST_pos_x', 'RIGHT_WRIST_pos_y', 'LEFT_HIP_pos_x', 'LEFT_HIP_pos_y', 'RIGHT_HIP_pos_x', 'RIGHT_HIP_pos_y', 'LEFT_ELBOW_LEFT_SHOULDER_LEFT_WRIST_angle', 'RIGHT_ELBOW_RIGHT_SHOULDER_RIGHT_WRIST_angle', 'LEFT_SHOULDER_LEFT_HIP_LEFT_ELBOW_angle', 'RIGHT_SHOULDER_RIGHT_HIP_RIGHT_ELBOW_angle']


In [21]:
df.head()

,video,frame,label,sports_ball_positions,LEFT_SHOULDER_vel,RIGHT_SHOULDER_vel,LEFT_SHOULDER_acc,RIGHT_SHOULDER_acc,LEFT_ELBOW_vel,RIGHT_ELBOW_vel,...,RIGHT_WRIST_pos_x,RIGHT_WRIST_pos_y,LEFT_HIP_pos_x,LEFT_HIP_pos_y,RIGHT_HIP_pos_x,RIGHT_HIP_pos_y,LEFT_ELBOW_LEFT_SHOULDER_LEFT_WRIST_angle,RIGHT_ELBOW_RIGHT_SHOULDER_RIGHT_WRIST_angle,LEFT_SHOULDER_LEFT_HIP_LEFT_ELBOW_angle,RIGHT_SHOULDER_RIGHT_HIP_RIGHT_ELBOW_angle
0,IMG_8185.MOV,1,Good_Shots,"619.95,1459.55",0.00,0.00,0.00,0.00,0.00,0.00,...,-124.50,-145.59,71.00,0.74,-71.00,-0.74,88.89,152.84,24.90,15.25
1,IMG_8185.MOV,2,Good_Shots,0,182.36,132.95,0.00,0.00,535.29,338.59,...,-119.78,-147.27,66.93,5.82,-66.93,-5.82,93.89,144.14,21.87,18.17
2,IMG_8185.MOV,3,Good_Shots,"602.58,1466.15",210.34,189.47,840.94,1698.71,240.73,145.91,...,-136.87,-146.80,66.00,0.62,-66.00,-0.62,92.06,153.75,20.16,17.65
3,IMG_8185.MOV,4,Good_Shots,"586.7,1456.39",275.77,304.77,1966.50,3465.34,274.25,398.70,...,-121.62,-125.83,66.47,-0.68,-66.47,0.68,91.41,156.46,21.12,14.65
4,IMG_8185.MOV,5,Good_Shots,"568.96,1446.75",484.78,283.40,6281.79,-642.27,333.34,245.84,...,-144.53,-148.36,66.96,-1.12,-66.96,1.12,90.14,168.70,19.26,15.01


In [22]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 56 entries, 0 to 55
Data columns (total 40 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   video                                         56 non-null     object 
 1   frame                                         56 non-null     int64  
 2   label                                         56 non-null     object 
 3   sports_ball_positions                         56 non-null     object 
 4   LEFT_SHOULDER_vel                             56 non-null     float64
 5   RIGHT_SHOULDER_vel                            56 non-null     float64
 6   LEFT_SHOULDER_acc                             56 non-null     float64
 7   RIGHT_SHOULDER_acc                            56 non-null     float64
 8   LEFT_ELBOW_vel                                56 non-null     float64
 9   RIGHT_ELBOW_vel                               56 non-null     float

In [23]:
df.columns.tolist()

['video',
 'frame',
 'label',
 'sports_ball_positions',
 'LEFT_SHOULDER_vel',
 'RIGHT_SHOULDER_vel',
 'LEFT_SHOULDER_acc',
 'RIGHT_SHOULDER_acc',
 'LEFT_ELBOW_vel',
 'RIGHT_ELBOW_vel',
 'LEFT_ELBOW_acc',
 'RIGHT_ELBOW_acc',
 'LEFT_WRIST_vel',
 'RIGHT_WRIST_vel',
 'LEFT_WRIST_acc',
 'RIGHT_WRIST_acc',
 'LEFT_HIP_vel',
 'RIGHT_HIP_vel',
 'LEFT_HIP_acc',
 'RIGHT_HIP_acc',
 'LEFT_SHOULDER_pos_x',
 'LEFT_SHOULDER_pos_y',
 'RIGHT_SHOULDER_pos_x',
 'RIGHT_SHOULDER_pos_y',
 'LEFT_ELBOW_pos_x',
 'LEFT_ELBOW_pos_y',
 'RIGHT_ELBOW_pos_x',
 'RIGHT_ELBOW_pos_y',
 'LEFT_WRIST_pos_x',
 'LEFT_WRIST_pos_y',
 'RIGHT_WRIST_pos_x',
 'RIGHT_WRIST_pos_y',
 'LEFT_HIP_pos_x',
 'LEFT_HIP_pos_y',
 'RIGHT_HIP_pos_x',
 'RIGHT_HIP_pos_y',
 'LEFT_ELBOW_LEFT_SHOULDER_LEFT_WRIST_angle',
 'RIGHT_ELBOW_RIGHT_SHOULDER_RIGHT_WRIST_angle',
 'LEFT_SHOULDER_LEFT_HIP_LEFT_ELBOW_angle',
 'RIGHT_SHOULDER_RIGHT_HIP_RIGHT_ELBOW_angle']